# Dask Out-of-Core S3 Validation

Validate **distributed** parquet reads from the air-gap object store and show that
workers perform the I/O — not the Jupyter kernel.

## Data layout (OTEL_Data_Generator)

```text
s3://$S3_BUCKET/$PREFIX/spans/date=YYYY-MM-DD/*.parquet
```

`date=` only (no `hour=`). Config comes from JupyterHub env (converge → zarf) via
`cluster_env` — no hard-coded secrets.

## The O(files) wall (why naive loads feel “hung”)

Silent **client-side** grind (Dask dashboard empty until the first task renders):

1. Recursive listing of every partition directory (sequential LISTs, high latency)
2. Graph construction / optimization — often **one task per file**
3. Shipping that graph to the scheduler

At a few dozen files this is invisible. At months × many files/day you can sit for
**minutes** before workers start; then tiny-file task overhead dominates.

**`filters=` is not enough** — it prunes *reads after* full discovery. Narrowing the
root prunes the **listing itself**.

### Near-term cures (this notebook)

| Cure | Practice |
|------|----------|
| Prune **before** discovery | `date="2026-07-01"` or `under="…/spans/date=…"` |
| List **once**, explicitly | `fs.find` / `list_span_parquet_keys` → file list → `read_parquet` (otel-navigator pattern) |
| Fatten tasks | `aggregate_files=True` |
| Iterate | `ddf.persist()` once, then many aggregates |

### Strategic cure (roadmap #42–45)

Object-store listing as a query planner is **O(files)**. Iceberg / the metadata plane
replaces it with **manifest-based planning** (catalog already knows files, partitions,
column stats). This notebook grind on ~20 days of spans is the small-scale preview of
petabyte HDF5-derived data — why pointer-table / kerchunk / Iceberg exist.

Editable copy: `cp /root/sample-notebooks/Dask_S3_Validation.ipynb /root/`


## 0. Imports + cluster config


In [ ]:
import os, sys, time, uuid
from datetime import datetime, timezone

os.environ.setdefault("BOKEH_RESOURCES", "inline")

import numpy as np
import pandas as pd
import s3fs
import dask
import dask.dataframe as dd
from dask.distributed import Client

for _p in ("/root/sample-notebooks", "/app", "/root"):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from cluster_env import (
    load_cluster_config,
    list_span_parquet_keys,
    load_active_spans_ddf,
    require_parquet_stack,
)

stack = require_parquet_stack()
print(f"dask={stack['dask']}  pyarrow={stack['pyarrow']}  backend={stack.get('parquet_backend')}")
CFG = load_cluster_config()
print(CFG.summary())


## 1. Connect to Dask


In [ ]:
client = CFG.connect_dask()
print("Dashboard:", client.dashboard_link)
print("Workers:", len(client.scheduler_info().get("workers", {})))
client


## 2. S3 client (same opts workers will use)

Path-style + timeouts required on custom endpoints. Listing here is **kernel-side** only
— workers need the same `storage_options` on `read_parquet`.


In [ ]:
o = CFG.storage_options()
fs = s3fs.S3FileSystem(**o)
bucket = CFG.s3_bucket
prefix = (CFG.otel_prefix or "otel-notebook").strip("/")
spans_key = f"{bucket}/{prefix}/spans"
print(f"endpoint set: {bool(CFG.s3_endpoint)}")
print(f"spans root:   s3://{spans_key}/")
# Cheap: list date= dirs only (one LIST), not every file
try:
    date_dirs = fs.ls(spans_key)
    print(f"date partitions ({len(date_dirs)}): {date_dirs[:8]}{'…' if len(date_dirs)>8 else ''}")
except Exception as e:
    print("ls spans failed:", e)


## 3. Load ddf — prune → list once → fatten

**Do not** call `dd.read_parquet("s3://…/spans/", filters=[("date", …)])` on this stack:
discovery still walks the whole tree first (dashboard empty).

Env knobs:
- `SPANS_DATE=2026-07-01` — prune listing to one day (recommended first run)
- `SPANS_DATE=` empty — full tree (expect slower client LIST + larger graph)


In [ ]:
# Prune BEFORE discovery (preferred)
SPANS_DATE = os.getenv("SPANS_DATE") or None  # e.g. "2026-07-01"
if SPANS_DATE == "":
    SPANS_DATE = None

print(f"SPANS_DATE={SPANS_DATE!r}  (None = entire spans/ tree)")
t0 = time.time()
keys = list_span_parquet_keys(CFG, date=SPANS_DATE)
t_list = time.time() - t0
print(f"listed {len(keys)} parquet in {t_list:.1f}s  sample={keys[:3]}")
if not keys:
    raise FileNotFoundError(
        f"no parquet under s3://{spans_key}/"
        + (f"date={SPANS_DATE}" if SPANS_DATE else "")
        + " — generator layout is date=YYYY-MM-DD/*.parquet. "
        "Run OTEL_Data_Generator first (same bucket/prefix/endpoint). "
        f"Expected s3://{spans_key}/date=*/batch_*.parquet"
    )
if len(keys) > 200:
    print(
        f"⚠ {len(keys)} files: client graph build can dominate. "
        f"Set SPANS_DATE to one day, or accept slower planning."
    )

# load_active_spans_ddf: same list + aggregate_files (fatten small files)
t1 = time.time()
ddf = load_active_spans_ddf(
    CFG,
    date=SPANS_DATE,
    aggregate_files=True,
    persist=False,  # set True if you will run many aggs below
)
print(f"lazy ddf ready in {time.time()-t1:.1f}s  partitions={ddf.npartitions}  cols={list(ddf.columns)[:12]}")
assert list(ddf.columns), "columns=[] — path/opts issue, not schema drift"
ddf


## 4. Prove workers (out-of-core parallel read)

Graph build is still **client**. Workers read data only on `.compute()`.

Watch the Dask dashboard during the next cell: multiple workers busy, memory bounded
vs full dataset size.


In [ ]:
# Optional: pay discovery+load once for the rest of the notebook
# ddf = ddf.persist()
# from dask.distributed import wait as dask_wait
# dask_wait(ddf)

n_workers = len(client.scheduler_info().get("workers", {}))
print(f"forcing distributed read on {n_workers} workers…")
t0 = time.time()
# Full pass on workers. Do NOT map_partitions(len).sum() — bare ints have no .sum()
# in dask-expr reductions (AttributeError: 'int' object has no attribute 'sum').
nrows = int(ddf.shape[0].compute())
print(f"rows≈{nrows:,}  in {time.time()-t0:.1f}s  partitions={ddf.npartitions}")

if "service_name" in ddf.columns:
    t1 = time.time()
    top = ddf.groupby("service_name").size().nlargest(15).compute()
    print(f"top services in {time.time()-t1:.1f}s:\n{top}")
elif "duration_ns" in ddf.columns:
    print("duration_ns mean ms:", (ddf["duration_ns"].mean().compute() / 1e6))

# Worker memory snapshot (evidence of bounded mem)
for addr, w in list(client.scheduler_info()["workers"].items())[:6]:
    mem = w.get("metrics", {}).get("memory", 0)
    print(f"  {addr.split('//')[-1][:28]:28} mem≈{mem/1e6:.0f} MiB")
print("✔ If dashboard showed multi-worker tasks and mem stayed << full data size → out-of-core OK")


## 5. Distributed aggregations

Reuse the same `ddf` (persist first if you skipped it above). **Never** `.compute()` the
full frame into the kernel.


In [ ]:
%%time
# Prefer columns that exist (OTEL generator schema varies slightly)
cols = list(ddf.columns)
print("columns:", cols)

if "service_name" in cols and "duration_ns" in cols:
    stats = (
        ddf.groupby("service_name")
        .agg({"duration_ns": ["count", "mean", "max"]})
        .compute()
    )
    if isinstance(stats.columns, pd.MultiIndex):
        stats.columns = ["_".join(map(str, c)).strip("_") for c in stats.columns]
    if "duration_ns_mean" in stats.columns:
        stats["mean_ms"] = stats["duration_ns_mean"] / 1e6
    display_cols = [c for c in stats.columns if "count" in c or "mean" in c or "max" in c]
    print(stats.sort_values(display_cols[0] if display_cols else stats.columns[0], ascending=False).head(15))
else:
    print(int(ddf.shape[0].compute()), "rows (no service_name)")


## 6. Optional: synthetic stress set (GENERATE_SYNTHETIC=1)

Default path uses **live** generator spans. Only enable synthetic if you need a
dedicated write path under `…/validation-dask/` (not the production prefix).


In [ ]:
GENERATE_SYNTHETIC = os.getenv("GENERATE_SYNTHETIC", "0").lower() in ("1", "true", "yes")
if not GENERATE_SYNTHETIC:
    print("skip synthetic (set GENERATE_SYNTHETIC=1 to write validation-dask/)")
else:
    TOTAL = int(os.getenv("VALIDATION_SPANS", "500000"))
    PER = int(os.getenv("VALIDATION_SPANS_PER_PART", "100000"))
    npart = max(1, (TOTAL + PER - 1) // PER)
    validation_path = f"{bucket}/{prefix}/validation-dask"
    print(f"synthetic → s3://{validation_path}/  spans={TOTAL} parts={npart}")

    def _gen(i, n=PER, services=20):
        services_l = [f"service-{j:02d}" for j in range(services)]
        idx = np.random.randint(0, services, n)
        base = int(datetime.now(timezone.utc).timestamp() * 1e9)
        return pd.DataFrame({
            "trace_id": [uuid.uuid4().hex for _ in range(n)],
            "span_id": [uuid.uuid4().hex[:16] for _ in range(n)],
            "service_name": np.array(services_l)[idx],
            "start_time_unix_nano": base - np.random.randint(0, 86400 * 10**9, n).astype(np.int64),
            "duration_ns": np.random.exponential(50_000_000, n).astype(np.int64),
            "status_code": np.random.choice(["OK", "ERROR", "UNSET"], n, p=[0.92, 0.05, 0.03]),
        })

    # Write with dask delayed / client map — still small tasks; OK for lab synthetic
    futures = []
    for i in range(npart):
        def _write(i=i):
            import pyarrow as pa
            import pyarrow.parquet as pq
            df = _gen(i)
            table = pa.Table.from_pandas(df, preserve_index=False)
            key = f"{validation_path}/part-{i:04d}.parquet"
            with fs.open(key, "wb") as f:
                pq.write_table(table, f)
            return key
        futures.append(client.submit(_write))
    written = client.gather(futures)
    print("wrote", len(written), "parts")
    uris = [f"s3://{k}" if not str(k).startswith("s3://") else k for k in written]
    ddf = dd.read_parquet(uris, storage_options=o, aggregate_files=True)
    print("synthetic ddf partitions", ddf.npartitions)


## 7. Cleanup

Do not delete live `otel-notebook/spans` data. Synthetic path only if you created it.


In [ ]:
# client.close()  # optional
print("done — re-run section 3 with a wider SPANS_DATE only after one-day path is proven")


## Summary

| Check | How you know |
|-------|----------------|
| Config from env | `CFG.summary()` shows endpoint/bucket/prefix |
| Listing pruned | `list_span_parquet_keys(date=…)` finishes in seconds |
| Graph not O(files) thrash | partitions after `aggregate_files` ≪ raw file count when files are tiny |
| Workers do I/O | dashboard busy during `.compute()`; multi-worker mem rise |
| Out-of-core | worker mem ≪ full dataset expanded size |

**Near-term:** explicit list + prune + fatten. **Durable:** Iceberg/metadata plane (#42–45).
